# Rotina diária – Daily Experiência BR no Databricks

Rode as células em ordem. Cada passo pode ser repetido sem problema: o envio só manda arquivos novos ou alterados e
nunca apaga nada; o job só pode ter uma execução por vez (uma segunda fica na fila).

1. **Google Sheets** – confere se a exportação de hoje (Apps Script, por volta das 9h) já chegou ao `G:\`.
2. **Envio** – manda ao volume os arquivos novos: brutos do MKTZap, planilhas do Drive e Google Sheets.
3. **Job** – dispara o processamento no Databricks (bronze → silver → gold, uns 15 minutos) e acompanha.
4. **Conferência** – mostra as tabelas gold e quando foram atualizadas.
5. **CSAT** – exporta a gold do CSAT para a pasta do Drive de onde o painel em Apps Script importa (às 10h).
6. **Tableau Cloud** – (só quando uma fonte mudar) recria e publica as fontes do workbook.

In [ ]:
from datetime import datetime

import enviar_arquivos as rotina

cfg = rotina.carregar_config()
w = rotina.cliente(cfg)
print("Conectado ao Databricks como", w.current_user.me().user_name)

## 1. As Google Sheets de hoje já foram exportadas?

In [ ]:
pronta, mensagem = rotina.verificar_google_sheets(cfg)
print(mensagem)

## 2. Envio dos arquivos

Se a célula anterior ainda não mostrou ✅, espere alguns minutos e rode-a de novo. Para ver o que seria enviado sem
enviar, use `rotina.enviar(cfg, simular=True)`.

In [ ]:
resultados = rotina.enviar(cfg, w)
contagem = rotina.imprimir_resultados(cfg, resultados)

## 3. Processamento no Databricks

In [ ]:
if contagem["erro"]:
    raise RuntimeError("Houve erro no envio (veja a célula anterior); o job não foi disparado.")
run_id = rotina.disparar_job(w, cfg["destino"]["job"])

In [ ]:
# Mostra o andamento de cada tarefa a cada 30 segundos até o job terminar (pode interromper e rodar de novo depois)
rotina.acompanhar_job(w, run_id)

## 4. Conferência das tabelas gold

In [ ]:
import pandas as pd

pd.DataFrame(rotina.conferir_gold(w, cfg), columns=["tabela", "linhas", "última gravação"])

## 5. CSAT – exportação para o painel em Apps Script

Depois do job: grava os CSV em `Fontes de Dados\auto\csat` (só os que mudaram). O Apps Script da planilha Painel C-SAT importa às 10h. Detalhes em `logic-local\csat\README.md`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(rotina.__file__).resolve().parents[2] / "csat" / "coleta"))
import exportar_sheets

# Para conferir sem gravar nada: exportar_sheets.exportar(cfg, w, simular=True)
resumo_csat = exportar_sheets.exportar(cfg, w)

## 6. Fontes no Tableau Cloud

Só quando uma fonte for nova ou mudar de colunas: as atualizações diárias já ficam agendadas no próprio Tableau Cloud.

In [ ]:
import publicar_tableau

# Só na primeira vez (ou ao trocar um token): publicar_tableau.guardar_tokens(cfg)
# Para só gerar e conferir as linhas, sem publicar: sem_publicar=True; para algumas fontes: somente=["HC Exp Brasil"]
resultados_tableau = publicar_tableau.publicar(cfg, w)
erros = [r for r in resultados_tableau if r[4]]
print(f"\n{len(resultados_tableau) - len(erros)} de {len(resultados_tableau)} fontes publicadas.")